# Homework 4
-   **Name:**  Victor Hugo Gomez Soto 
-  **e-mail:** -- victor.gomez2701@alumnos.udg.mx --


# MODULES

In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.spatial import distance
from scipy.stats import wrapcauchy, levy_stable
import math
import dash
from dash import dcc, html
from dash.dependencies import Input, Output



In [4]:
# Nota: Esta clase la importaremos junto con el segundo bloque de modulos
################# http://www.pygame.org/wiki/2DVectorClass ##################
class Vec2d(object):
    """2d vector class, supports vector and scalar operators,
       and also provides a bunch of high level functions
       """
    __slots__ = ['x', 'y']

    def __init__(self, x_or_pair, y = None):
        if y == None:            
            self.x = x_or_pair[0]
            self.y = x_or_pair[1]
        else:
            self.x = x_or_pair
            self.y = y
            
    # Addition
    def __add__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x + other.x, self.y + other.y)
        elif hasattr(other, "__getitem__"):
            return Vec2d(self.x + other[0], self.y + other[1])
        else:
            return Vec2d(self.x + other, self.y + other)

    # Subtraction
    def __sub__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x - other.x, self.y - other.y)
        elif (hasattr(other, "__getitem__")):
            return Vec2d(self.x - other[0], self.y - other[1])
        else:
            return Vec2d(self.x - other, self.y - other)
    
    # Vector length
    def get_length(self):
        return math.sqrt(self.x**2 + self.y**2)
    
    # rotate vector
    def rotated(self, angle):        
        cos = math.cos(angle)
        sin = math.sin(angle)
        x = self.x*cos - self.y*sin
        y = self.x*sin + self.y*cos
        return Vec2d(x, y)
     # Método para convertir el vector en una tupla
    def to_tuple(self):
        return (self.x, self.y)

In [ ]:
#####################################################################################
# Brownian motion trajectoy
#####################################################################################
def bm_2d(n_steps=1000, speed=5, start_pos=(0, 0)):    
    pos = Vec2d(*start_pos)
    trajectory = [pos.to_tuple()]
    
    for _ in range(n_steps):
        turn_angle = np.random.uniform(low=-np.pi, high=np.pi)
        step = Vec2d(speed, 0).rotated(turn_angle)
        pos += step
        trajectory.append(pos.to_tuple())
    
    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    return df
#####################################################################################
# Correlated Random Walk 
#####################################################################################
def rw_2d(n_steps=1000, speed=5, start_pos=(0, 0), c=0.5):   
    pos = Vec2d(*start_pos)
    trajectory = [pos.to_tuple()]
    
    angle = 0
    for _ in range(n_steps):
        delta_angle = wrapcauchy.rvs(c)
        angle += delta_angle
        step = Vec2d(speed, 0).rotated(angle)
        pos += step
        trajectory.append(pos.to_tuple())
    
    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    return df

#####################################################################################
# Levi Flight  
#####################################################################################
def levy_flight(n_steps=1000, alpha=1.5, scale=1.0, c=0.5, start_pos=(0, 0)):
    pos = Vec2d(*start_pos)
    trajectory = [pos.to_tuple()]
    angle = 0  

    for _ in range(n_steps):
        step_size = np.abs(levy_stable.rvs(alpha, 0, scale=scale))
        delta_angle = wrapcauchy.rvs(c)  
        angle += delta_angle
        step = Vec2d(step_size, 0).rotated(angle)
        pos += step
        trajectory.append(pos.to_tuple())

    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    return df

#####################################################################################
# path length 
#####################################################################################
def path_length(df):
    """Calcula la longitud total del camino recorrido."""
    distances = np.sqrt(np.diff(df["x_pos"])**2 + np.diff(df["y_pos"])**2)
    return np.sum(distances)


In [ ]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objects as go
import pandas as pd
import numpy as np

class Vec3d:
    """3D vector class for trajectory simulation"""
    __slots__ = ['x', 'y', 'z']

    def __init__(self, x, y, z):
        self.x, self.y, self.z = x, y, z

    def __add__(self, other):
        return Vec3d(self.x + other.x, self.y + other.y, self.z + other.z)

    def rotated(self, angle_xy, angle_xz):
        """Rotar en los planos XY y XZ"""
        cos_xy, sin_xy = math.cos(angle_xy), math.sin(angle_xy)
        cos_xz, sin_xz = math.cos(angle_xz), math.sin(angle_xz)

        # Rotación en el plano XY
        x_new = self.x * cos_xy - self.y * sin_xy
        y_new = self.x * sin_xy + self.y * cos_xy

        # Rotación en el plano XZ
        z_new = self.z * cos_xz - self.x * sin_xz
        x_new = self.x * cos_xz + self.z * sin_xz  # Corregir x después de la rotación en XZ

        return Vec3d(x_new, y_new, z_new)

    def to_tuple(self):
        return (self.x, self.y, self.z)
#####################################################################################
# Brownian motion trajectoy
#####################################################################################
def bm_3d(n_steps=1000, speed=5, start_pos=(0, 0, 0)):
    pos = Vec3d(*start_pos)
    trajectory = [pos.to_tuple()]

    for _ in range(n_steps):
        turn_angle_xy = np.random.uniform(-np.pi, np.pi)
        turn_angle_xz = np.random.uniform(-np.pi, np.pi)
        step = Vec3d(speed, 0, 0).rotated(turn_angle_xy, turn_angle_xz)
        pos += step
        trajectory.append(pos.to_tuple())

    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos", "z_pos"])
    return df
#####################################################################################
# Correlated Random Walk 
#####################################################################################
def rw_3d(n_steps=1000, speed=5, start_pos=(0, 0, 0), c=0.5):   
    pos = Vec3d(*start_pos)
    trajectory = [pos.to_tuple()]

    angle_xy = 0  # Ángulo en el plano XY
    angle_xz = 0  # Ángulo en el plano XZ

    for _ in range(n_steps):
        delta_angle_xy = np.random.vonmises(mu=0, kappa=c)  # Variación del ángulo XY
        delta_angle_xz = np.random.vonmises(mu=0, kappa=c)  # Variación del ángulo XZ

        angle_xy += delta_angle_xy
        angle_xz += delta_angle_xz

        step = Vec3d(speed, 0, 0).rotated(angle_xy, angle_xz)  # Movimiento en 3D
        pos += step
        trajectory.append(pos.to_tuple())

    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos", "z_pos"])
    return df

#####################################################################################
# Levi Flight  
#####################################################################################
def levy_flight_3d(n_steps=1000, alpha=1.5, scale=1.0, c=0.5, start_pos=(0, 0, 0)):
    pos = Vec3d(*start_pos)  # Iniciar en 3D
    trajectory = [pos.to_tuple()]
    
    angle_xy = 0  # Ángulo en el plano XY
    angle_xz = 0  # Ángulo en el plano XZ

    for _ in range(n_steps):
        step_size = np.abs(levy_stable.rvs(alpha, 0, scale=scale))  # Paso de Lévy
        delta_angle_xy = np.random.vonmises(mu=0, kappa=c)  # Variación del ángulo en XY
        delta_angle_xz = np.random.vonmises(mu=0, kappa=c)  # Variación del ángulo en XZ

        angle_xy += delta_angle_xy
        angle_xz += delta_angle_xz

        step = Vec3d(step_size, 0, 0).rotated(angle_xy, angle_xz)  # Movimiento en 3D
        pos += step
        trajectory.append(pos.to_tuple())

    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos", "z_pos"])
    
    return df

#####################################################################################
# path length 
#####################################################################################
def path_length_3d(trajectory):    
    # Obtener la distancia euclidiana entre cada par de puntos consecutivos en 3D
    distances = np.array([
        distance.euclidean(trajectory.iloc[i - 1], trajectory.iloc[i])
        for i in range(1, trajectory.shape[0])
    ])

    # Devolver la suma acumulativa de las distancias
    return np.cumsum(distances)
#####################################################################################
# turning angle distribution
#####################################################################################

def turning_angle_distribution_3d(df):
    
    # Diferencias de posición en X, Y y Z
    dx = df["x_pos"].diff().values[1:]  # Omitimos el primer valor NaN
    dy = df["y_pos"].diff().values[1:]
    dz = df["z_pos"].diff().values[1:]

    # Construcción de los vectores de movimiento en 3D
    v1 = np.column_stack((dx[:-1], dy[:-1], dz[:-1]))  # Primeros desplazamientos
    v2 = np.column_stack((dx[1:], dy[1:], dz[1:]))  # Segundos desplazamientos

    # Normalizamos los vectores
    norm_v1 = np.linalg.norm(v1, axis=1)
    norm_v2 = np.linalg.norm(v2, axis=1)

    # Evitar división por cero
    valid_indices = (norm_v1 > 0) & (norm_v2 > 0)
    v1, v2 = v1[valid_indices], v2[valid_indices]
    norm_v1, norm_v2 = norm_v1[valid_indices], norm_v2[valid_indices]

    # Producto punto y ángulos en 3D
    dot_product = np.einsum("ij,ij->i", v1, v2)
    cos_theta = dot_product / (norm_v1 * norm_v2)  # Cálculo del coseno del ángulo
    cos_theta = np.clip(cos_theta, -1, 1)  # Evitar errores numéricos fuera de [-1,1]

    angles = np.arccos(cos_theta)  # Convertir a ángulos en radianes
    return np.degrees(angles)  # Convertir a grados

#####################################################################################
# mean squared displacement
#####################################################################################
def mean_squared_displacement_3d(df):

    x = df['x_pos'].values
    y = df['y_pos'].values
    z = df['z_pos'].values  # Se agrega la tercera dimensión
    N = len(x)
    msd = np.zeros(N)

    for t in range(N):
        dx = x[t:] - x[:N-t]  # Desplazamiento en X
        dy = y[t:] - y[:N-t]  # Desplazamiento en Y
        dz = z[t:] - z[:N-t]  # Desplazamiento en Z
        squared_displacement = dx**2 + dy**2 + dz**2  # MSD = dx² + dy² + dz²
        msd[t] = np.mean(squared_displacement)

    return msd

# Inicializar la app Dash
app = dash.Dash(__name__)

# Definir función para generar una trayectoria
def generate_trajectory(traj_type='BM', n_steps=500, speed=5, alpha=1.5, scale=1.0, c=0.5):
    if traj_type == 'BM':
        df = bm_3d(n_steps, speed)
    elif traj_type == 'CRW':
        df = rw_3d(n_steps, speed, c=c)
    else:
        df = levy_flight_3d(n_steps, alpha, scale, c)
    return df

# Estilos para mejorar la UI
styles = {
    'container': {
        'width': '80%',
        'margin': 'auto',
        'padding': '20px',
        'fontFamily': 'Arial, sans-serif',
        'backgroundColor': '#f8f9fa',
        'borderRadius': '10px',
        'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)'
    },
    'header': {
        'textAlign': 'center',
        'fontSize': '24px',
        'fontWeight': 'bold',
        'marginBottom': '20px'
    },
    'panel': {
        'padding': '15px',
        'border': '1px solid #ccc',
        'borderRadius': '5px',
        'backgroundColor': '#fff',
        'marginBottom': '15px'
    }
}

# Layout de la app
app.layout = html.Div(style=styles['container'], children=[
    html.H1("Simulación de Trayectorias Aleatorias", style=styles['header']),
    
    html.Div(style=styles['panel'], children=[
        html.Label("Selecciona el tipo de trayectoria:"),
        dcc.RadioItems(
            id='traj-selector',
            options=[
                {'label': 'Movimiento Browniano (BM)', 'value': 'BM'},
                {'label': 'Camino Aleatorio Correlacionado (CRW)', 'value': 'CRW'},
                {'label': 'Vuelo de Lévy (LF)', 'value': 'LF'}
            ],
            value='BM',
            inline=True
        )
    ]),

    html.Div(style=styles['panel'], children=[
        html.Label("Número de pasos:"),
        dcc.Slider(id='n-steps', min=100, max=1000, step=100, value=500, 
                   marks={i: str(i) for i in range(100, 1100, 200)})
    ]),

    html.Div(style=styles['panel'], children=[
        html.Label("Velocidad:"),
        dcc.Slider(id='speed', min=1, max=10, step=1, value=5)
    ]),

    html.Div(id='extra-params', style=styles['panel']),

    html.Div(style=styles['panel'], children=[
        html.Label("Selecciona la métrica a visualizar:"),
        dcc.Dropdown(
            id='metric-selector',
            options=[
                {'label': 'Path Length (PL)', 'value': 'PL'},
                {'label': 'Mean Squared Displacement (MSD)', 'value': 'MSD'},
                {'label': 'Turning Angle Distribution (TAD)', 'value': 'TAD'}
            ],
            value='PL'
        )
    ]),

    dcc.Graph(id='trajectory-plot'),
    dcc.Graph(id='metric-plot')
])

# Callback para mostrar parámetros adicionales
@app.callback(
    Output('extra-params', 'children'),
    [Input('traj-selector', 'value')]
)
def update_params(traj_type):
    if traj_type == 'BM':
        return ''
    return html.Div([
        html.Label("Coeficiente de Cauchy (CRW & LF):"),
        dcc.Slider(id='c-coefficient', min=0.1, max=1, step=0.1, value=0.5),
        html.Label("Exponente de Lévy (LF):"),
        dcc.Slider(id='alpha', min=0.5, max=2, step=0.1, value=1.5),
        html.Label("Escala (LF):"),
        dcc.Slider(id='scale', min=0.1, max=5, step=0.1, value=1.0)
    ])

# Callback para actualizar gráficos
@app.callback(
    [Output('trajectory-plot', 'figure'),
     Output('metric-plot', 'figure')],
    [Input('traj-selector', 'value'),
     Input('n-steps', 'value'),
     Input('speed', 'value'),
     Input('metric-selector', 'value')]
)
def update_plots(traj_type, n_steps, speed, metric):
    df = generate_trajectory(traj_type, n_steps, speed)  # Ahora usa la versión 3D

    # print(f"Path Length for {traj_type}: {path_length(df)}")

    # 🔷 **Gráfica de trayectoria en 3D**
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter3d(
        x=df['x_pos'], 
        y=df['y_pos'], 
        z=df['z_pos'],  # Ahora agregamos la tercera dimensión
        mode='lines', 
        name='Trayectoria',
        line=dict(width=3)  # Línea más visible
    ))

    fig1.update_layout(
        title='Trayectoria en 3D',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        )
    )

    # 🔷 **Gráfica de métrica seleccionada**
    if metric == 'PL':
        metric_value = path_length(df)
        print(f"Path Length for {traj_type}: {path_length(df)}")
        
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(
            x=np.arange(len(metric_value)),  # El índice como eje X (tiempo)
            y=metric_value,  
            mode='lines',
            name='Path Length'
        ))
        
        fig2.update_layout(
            title='Path Length',
            xaxis_title='Tiempo',
            yaxis_title='Distancia Acumulada',
            bargap=0.5  # Espaciado entre barras (no aplica aquí, pero mantiene el formato)
        )
    
    elif metric == 'MSD':
        metric_values = mean_squared_displacement_3d(df)
        
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(y=metric_values, mode='lines', name='MSD'))
        fig2.update_layout(title='Mean Squared Displacement')

    else:
        metric_values = turning_angle_distribution_3d(df)
        hist, bin_edges = np.histogram(metric_values, bins=30, density=True)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])  

        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(
            x=bin_centers, 
            y=hist, 
            mode='lines', 
            name="Densidad de Ángulos", 
            line=dict(color='blue')
        ))

        fig2.update_layout(
            title='Turning Angle Distribution',
            xaxis_title="Ángulo (grados)",
            yaxis_title="Densidad"
        )

    return fig1, fig2


# Ejecutar la aplicación
if __name__ == '__main__':
    app.run_server(debug=True)


Path Length for BM: [   6.77910525   13.60987747   18.64133066   25.44033153   32.36727975
   37.55374935   44.54664735   49.71485617   56.17826781   63.21929992
   69.91695505   75.32728843   80.81492829   86.17453918   92.31475687
   97.65682078  104.53674918  110.22045625  117.2777873   124.30796935
  131.37713746  136.49412255  143.47718212  150.41295845  157.28786673
  163.92506649  170.98903382  178.0377422   183.05632401  188.07683717
  193.99940832  199.37012562  206.38914071  213.31231755  218.36392458
  225.23850828  232.30072772  239.25670666  246.32756734  251.95749539
  258.30626362  263.49727302  268.93286519  273.94024218  280.57999527
  287.2896815   294.04822061  300.77284998  305.78143726  312.80831307
  319.87534772  324.95315316  331.84490107  338.90172924  345.89624704
  352.75736174  357.77695677  363.39992506  370.14874647  375.18176583
  382.00709325  387.10568868  392.54287712  397.60769643  402.74981721
  407.80996678  414.86474591  420.65253124  427.56078787 